# 102 · Framework: Sessions, Projectors, and Writers

In the previous tutorial, we manually interacted with the `Ledger`. While powerful, manual event management becomes complex as your scientific logic grows. 

The `EarlySign` **Framework layer** introduces the **Pattern G** architecture, which provides three core abstractions to manage the lifecycle of an analysis:

1. **`Session`**: Defines a "scientific horizon"—a bounded context and time window for your analysis.
2. **`Projector`**: Reconstructs structured state from raw events (Read Model).
3. **`Writer`**: Commits analytical facts back to the ledger with automatic lineage tracking (Write Model).

In this tutorial, we will build a simple "Counter" system using these abstractions.

## 1. Setup

We initialize a Ledger as before.

In [ ]:
import ibis
from pydantic import BaseModel
from earlysign.core.ledger import Ledger
from earlysign.v1.framework.session import Session
from earlysign.v1.framework.projector import Projector, ProjectionResult
from earlysign.v1.framework.trace import TraceId

con = ibis.connect("duckdb://:memory:")
ledger = Ledger(con, "framework_demo").bind(experiment_id="102_demo")
ledger.ensure()

## 2. Defining a Projector

A `Projector` is a scientific lens. It takes a stream of events and "projects" them into a meaningful object. Crucially, it returns a **`ProjectionResult`** containing both the data and the **`trace`** (the UUIDs of the source events).

In [ ]:
class CounterState(BaseModel):
    total: int = 0

class IncrementProjector(Projector[CounterState]):
    """Sum up all 'Increment' events to get current state."""
    def project(self, table: ibis.Expr) -> ProjectionResult[CounterState]:
        # 1. Select relevant events
        matches = table.filter(table.type == "Increment")
        pdf = matches.execute()
        
        if pdf.empty:
            return ProjectionResult(data=CounterState(total=0), trace=[])
        
        # 2. Derive state
        total = pdf['payload'].apply(lambda x: x['value']).sum()
        
        # 3. Collect lineage (trace)
        trace = [TraceId(str(u)) for u in pdf['uuid']]
        
        return ProjectionResult(data=CounterState(total=total), trace=trace)

## 3. Reading within a Session

We use **`sess.read()`** to evoke a projector. The session automatically aggregates the traces of every read operation, building a global lineage for the current analysis.

In [ ]:
# Insert some raw data first
ledger.insert(type="Increment", data={"value": 10})
ledger.insert(type="Increment", data={"value": 5})

with Session(ledger) as sess:
    # Evoke the projector
    result = sess.read(IncrementProjector())
    
    print(f"Current State: {result.data.total}")
    print(f"Session Trace (Provenance): {sess.trace}")

## 4. Committing Facts (Writers)

When we reach a conclusion or perform an expensive calculation, we commit it back to the ledger. By using **`sess.commit()`**, the new event is automatically tagged with the accumulated session trace. 

This creates an "Implicit Web of Proof"—you can always see exactly which increments influenced which final report.

In [ ]:
class FinalReport(BaseModel):
    summary_text: str

with Session(ledger) as sess:
    # 1. Read current state
    state = sess.read(IncrementProjector()).data
    
    # 2. Derive a new fact
    conclusion = FinalReport(summary_text=f"The final count reached {state.total}.")
    
    # 3. Commit with automatic lineage
    sess.commit(conclusion)

# Look at the ledger to see the Trace in the 'metadata' column
display(ledger.t.execute())

## 5. Summary

- **`Session`**: Orchestrates the analysis and tracks lineage.
- **`Projector`**: Translates raw history into scientific objects.
- **`sess.commit`**: Persists analytical results while preserving the "web of proof".

In the next section, we will use these building blocks to implement a real-world **Group Sequential Design** template.